In [1]:
import os
import yaml
import pandas as pd
import numpy as np

with open("../configs/project_config.yaml") as f:
    config = yaml.safe_load(f)

EXAMPLES_PATH = "../esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet"
PRODUCTS_PATH = "../esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet"

examples = pd.read_parquet(EXAMPLES_PATH, filters=[("product_locale", "=", "us")])
examples = examples[examples["small_version"] == 1].reset_index(drop=True)

FULL_TO_CODE = {"E": "Exact", "S": "Substitute", "C": "Complement", "I": "Irrelevant"}
LABEL_MAP = config["data"]["label_mapping"]
examples["relevance"] = examples["esci_label"].map(FULL_TO_CODE).map(LABEL_MAP)
assert examples["relevance"].isnull().sum() == 0

judged_product_ids = examples["product_id"].unique()
products_us = pd.read_parquet(PRODUCTS_PATH, filters=[("product_locale", "=", "us")])
products = products_us[products_us["product_id"].isin(judged_product_ids)].reset_index(drop=True)
del products_us

def build_product_text(row):
    parts = []
    missing = []
    for field, label in [
        ("product_title", "title"),
        ("product_brand", "brand"),
        ("product_bullet_point", "bullets"),
        ("product_color", "color"),
    ]:
        val = row[field]
        if pd.notna(val) and str(val).strip():
            parts.append(f"{label}: {val}")
        else:
            missing.append(field)
    return " [SEP] ".join(parts), missing

results = products.apply(build_product_text, axis=1)
products["product_text"] = results.apply(lambda r: r[0])
missing_counts = pd.Series([f for r in results for f in r[1]]).value_counts()
print("missing-field counts across catalogue:")
print(missing_counts)

os.makedirs("../data/processed", exist_ok=True)
products.to_parquet("../data/processed/products.parquet", index=False)
examples.to_parquet("../data/processed/judgments.parquet", index=False)
print("saved products.parquet:", products.shape)
print("saved judgments.parquet:", examples.shape)

products[["product_id", "product_title", "product_text"]].sample(3, random_state=42)


missing-field counts across catalogue:
product_color           155144
product_bullet_point     63627
product_brand            27191
Name: count, dtype: int64
saved products.parquet: (482105, 8)
saved judgments.parquet: (601354, 10)


,product_id,product_title,product_text
4215,B08KTP9JQP,Ace Tech Cellular LCD Screen Replacement Compa...,title: Ace Tech Cellular LCD Screen Replacemen...
137078,1495488594,The Everything Pet Rabbit Handbook: Your Ultim...,title: The Everything Pet Rabbit Handbook: You...
205595,B00APL0856,BW Technologies XT-XWHM-Y-NA GasAlertMax XT II...,title: BW Technologies XT-XWHM-Y-NA GasAlertMa...


In [2]:
val_ids_df = pd.read_csv("../data/processed/splits/validation_query_ids.csv")
val_query_ids = set(val_ids_df["query_id"])
print("validation queries:", len(val_query_ids))

val_examples = examples[examples["query_id"].isin(val_query_ids)].copy()
print("validation judged pairs:", len(val_examples))

val_queries = val_examples.drop_duplicates("query_id")[["query_id", "query"]].reset_index(drop=True)

relevant_lookup = (
    val_examples[val_examples["relevance"] >= 1]
    .groupby("query_id")["product_id"]
    .apply(set)
    .to_dict()
)
print("queries with at least 1 relevant product:", len(relevant_lookup))
print("queries with zero relevant products:", len(val_query_ids) - len(relevant_lookup))


validation queries: 4477
validation judged pairs: 89411
queries with at least 1 relevant product: 4477
queries with zero relevant products: 0


In [4]:
import time
import bm25s

corpus_texts = products["product_text"].tolist()

t0 = time.time()
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords="en", show_progress=True)
print(f"tokenized in {time.time()-t0:.1f}s")

t0 = time.time()
retriever = bm25s.BM25(corpus=corpus_texts)
retriever.index(corpus_tokens)
print(f"indexed {len(corpus_texts)} products in {time.time()-t0:.1f}s")

product_id_array = products["product_id"].to_numpy()


Split strings:   0%|          | 0/482105 [00:00<?, ?it/s]

tokenized in 29.3s


BM25S Count Tokens:   0%|          | 0/482105 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/482105 [00:00<?, ?it/s]

indexed 482105 products in 24.1s


In [9]:
retriever.corpus = None  # return integer positions, not stored corpus text

query_texts = val_queries["query"].tolist()
query_tokens = bm25s.tokenize(query_texts, stopwords="en", show_progress=False)

t0 = time.time()
result_idx, result_scores = retriever.retrieve(query_tokens, k=100, show_progress=False)
elapsed = time.time() - t0

print(f"retrieved top100 for {len(query_texts)} queries in {elapsed:.1f}s -> {elapsed/len(query_texts)*1000:.1f}ms/query")
print("result_idx dtype:", result_idx.dtype, "shape:", result_idx.shape)


retrieved top100 for 4477 queries in 12.6s -> 2.8ms/query
result_idx dtype: int32 shape: (4477, 100)


In [10]:
latencies = []
for q in query_texts:
    t0 = time.time()
    q_tok = bm25s.tokenize([q], stopwords="en", show_progress=False)
    _ = retriever.retrieve(q_tok, k=100, show_progress=False)
    latencies.append(time.time() - t0)

latencies = np.array(latencies) * 1000  # ms

print(f"median: {np.median(latencies):.2f}ms")
print(f"p95: {np.percentile(latencies, 95):.2f}ms")
print(f"p99: {np.percentile(latencies, 99):.2f}ms")
print(f"max: {latencies.max():.2f}ms")


median: 2.57ms
p95: 4.95ms
p99: 6.26ms
max: 40.40ms


In [11]:
predicted_ids = product_id_array[result_idx]  # shape (4477, 100), row order matches val_queries

query_relevance = {
    qid: dict(zip(g["product_id"], g["relevance"]))
    for qid, g in val_examples.groupby("query_id")
}

def dcg(relevances):
    relevances = np.asarray(relevances, dtype=float)
    ranks = np.arange(1, len(relevances) + 1)
    return np.sum((2 ** relevances - 1) / np.log2(ranks + 1))

records = []
for i, row in val_queries.reset_index(drop=True).iterrows():
    qid = row["query_id"]
    preds = predicted_ids[i]
    rel_lookup = query_relevance[qid]

    retrieved_relevant = sum(1 for pid in preds if rel_lookup.get(pid, 0) >= 1)
    total_relevant = sum(1 for r in rel_lookup.values() if r >= 1)
    recall_100 = retrieved_relevant / total_relevant

    top10_rel = [rel_lookup.get(pid, 0) for pid in preds[:10]]
    ideal_rel = sorted(rel_lookup.values(), reverse=True)[:10]
    ndcg_10 = dcg(top10_rel) / dcg(ideal_rel) if dcg(ideal_rel) > 0 else 0.0

    mrr_10 = 0.0
    for rank, pid in enumerate(preds[:10], start=1):
        if rel_lookup.get(pid, 0) >= 1:
            mrr_10 = 1.0 / rank
            break

    records.append({"query_id": qid, "recall_100": recall_100, "ndcg_10": ndcg_10, "mrr_10": mrr_10})

metrics_df = pd.DataFrame(records)
print(metrics_df[["recall_100", "ndcg_10", "mrr_10"]].mean())
metrics_df.describe()


recall_100    0.460352
ndcg_10       0.310562
mrr_10        0.553209
dtype: float64


,query_id,recall_100,ndcg_10,mrr_10
count,4477.000000,4477.000000,4477.000000,4477.000000
mean,56162.521331,0.460352,0.310562,0.553209
std,34589.564914,0.320392,0.286632,0.439079
min,2.000000,0.000000,0.000000,0.000000
25%,25894.000000,0.181818,0.000000,0.000000
50%,56220.000000,0.437500,0.252795,0.500000
75%,86083.000000,0.750000,0.538680,1.000000
max,130539.000000,1.000000,1.000000,1.000000


In [12]:
import json

rank_records = []
for i, row in val_queries.reset_index(drop=True).iterrows():
    qid = row["query_id"]
    for rank, (pid, score) in enumerate(zip(predicted_ids[i], result_scores[i]), start=1):
        rank_records.append({"query_id": qid, "rank": rank, "product_id": pid, "bm25_score": score})

rankings_df = pd.DataFrame(rank_records)
os.makedirs("../artifacts", exist_ok=True)
rankings_df.to_parquet("../artifacts/bm25_validation_rankings.parquet", index=False)
print("saved rankings:", rankings_df.shape)

metrics_summary = {
    "n_queries": len(val_queries),
    "catalogue_size": len(products),
    "recall_100_mean": float(metrics_df["recall_100"].mean()),
    "ndcg_10_mean": float(metrics_df["ndcg_10"].mean()),
    "mrr_10_mean": float(metrics_df["mrr_10"].mean()),
    "recall_100_median": float(metrics_df["recall_100"].median()),
    "ndcg_10_median": float(metrics_df["ndcg_10"].median()),
    "mrr_10_median": float(metrics_df["mrr_10"].median()),
    "latency_ms_median": float(np.median(latencies)),
    "latency_ms_p95": float(np.percentile(latencies, 95)),
    "latency_ms_p99": float(np.percentile(latencies, 99)),
    "index_build_time_s": 33.4 + 25.6,
}

os.makedirs("../reports", exist_ok=True)
with open("../reports/bm25_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(json.dumps(metrics_summary, indent=2))


saved rankings: (447700, 4)
{
  "n_queries": 4477,
  "catalogue_size": 482105,
  "recall_100_mean": 0.4603518626716506,
  "ndcg_10_mean": 0.3105624680708995,
  "mrr_10_mean": 0.553209260027442,
  "recall_100_median": 0.4375,
  "ndcg_10_median": 0.25279507893794145,
  "mrr_10_median": 0.5,
  "latency_ms_median": 2.5670528411865234,
  "latency_ms_p95": 4.953241348266601,
  "latency_ms_p99": 6.255435943603511,
  "index_build_time_s": 59.0
}


In [13]:
metrics_labeled = metrics_df.merge(val_queries, on="query_id")

worst = metrics_labeled.sort_values("ndcg_10").head(10)
best = metrics_labeled.sort_values("ndcg_10", ascending=False).head(10)

title_lookup = dict(zip(products["product_id"], products["product_title"]))

def show_query(qid, query_text, n_preview=5):
    print(f"\nquery_id={qid}  query='{query_text}'")
    rel_lookup = query_relevance[qid]
    top_relevant = sorted(rel_lookup.items(), key=lambda x: -x[1])[:n_preview]
    print("  top judged-relevant products for this query:")
    for pid, rel in top_relevant:
        print(f"    [{rel}] {title_lookup.get(pid, '?')}")
    idx = val_queries.reset_index(drop=True).query("query_id == @qid").index[0]
    print("  top predicted (BM25):")
    for pid in predicted_ids[idx][:n_preview]:
        print(f"    [{rel_lookup.get(pid, 0)}] {title_lookup.get(pid, '?')}")

print("=" * 20, "WORST 10 (lowest NDCG@10)", "=" * 20)
for _, row in worst.iterrows():
    show_query(row["query_id"], row["query"])

print("\n" + "=" * 20, "BEST 10 (highest NDCG@10)", "=" * 20)
for _, row in best.iterrows():
    show_query(row["query_id"], row["query"])


==================== WORST 10 (lowest NDCG@10) ====================

query_id=38847  query='extra ordinary'
  top judged-relevant products for this query:
    [3] The Extraordinary Journey of the Fakir
    [3] The TNGA
    [3] Extraordinary: The Seeding
    [3] extraordinary: the stan romanek story
    [3] Extraordinary People: My Face is Killing Me
  top predicted (BM25):
    [0] The Ordinary The Daily Set (3 Pcs: The Ordinary Squalane Cleanser - The Ordinary Hyaluronic Acid 2% + B5 - The Ordinary Natural Moisturizing Factors + HA)
    [0] Colgate 360 Enamel Health Extra Soft Toothbrush for Sensitive Teeth
    [0] Rechargeable Light Bulbs, Back Up Battery Light Bulbs, 15W 80W Equivalent 6000K 1200mAh Battery Operated Light Bulb for Power Outage Hurricane, Emergency Lights for Home Power Failure, Pack of 2
    [0] The Ordinary 100% Plant-derived Squalane 30ml (Pack of 2)
    [0] The Ordinary Buffet 30ml

query_id=82709  query='pot container no smell'
  top judged-relevant products for 